# AI512 — Laboratory Sheet: Adversarial Search and Game Playing

**Course:** Artificial Intelligence (AI512)  
**Lab topic:** Minimax, Alpha–Beta Pruning, Nim, and Grundy's Game  
**Student Name:** Pranav Patel  
**Roll Number:** 2026AIM1016  
**Date:** 17-10-2026

## Learning objectives

By completing this lab, you should be able to:

- evaluate a deterministic, two-player game tree using Minimax;
- implement Alpha–Beta pruning and count evaluated terminal states;
- model standard Nim and identify winning moves;
- model Grundy's Game and evaluate legal pile-splitting moves;
- explain the difference between standard Nim and Grundy's Game.

## Instructions

1. Complete each function where `NotImplementedError` is raised.
2. Run the provided test cells and inspect the outputs.
3. Preserve the given state representations and move rules.
4. For game-tree algorithms, use the specified child order so results are reproducible.
5. Read the game rules and inspect the visual before attempting each part.
6. Read each task before the starter-code cell; implement only where indicated.
7. Include short explanations in the answer spaces. Do not submit code output without interpretation.


## Part 1 — Minimax Search

Minimax is used for deterministic, two-player, zero-sum games with perfect information. **MAX** chooses moves to maximize the utility, while **MIN** chooses moves to minimize it.

The tree has the structure `MAX → MIN → MAX → terminal utility`. The order of children is fixed in `GAME_TREE`; retain that order when evaluating the tree.

### Game-tree visual

```text
                         R (MAX)
                    /       |       \
                A (MIN)   B (MIN)   C (MIN)
                /   \     /   \     /   \
              A1    A2   B1    B2   C1    C2       (MAX)
             / \   / \  / \   / \  / \   / \
            3   5  6 9 1   2  0 -1 7  4 5   6  (terminal utilities)
```

**How to read it:** Start at `R`. MAX selects the child with the largest backed-up value. At the next level MIN selects the smallest child value. At the last level, the numbers are terminal utilities from MAX's point of view.


Read the task below, then complete the starter code in the next cell.

### Task 1 — Implement and analyse Minimax

Read the game-tree diagram above before coding.

1. Implement `minimax()` recursively.
2. Implement `minimax_decision()` to choose the best action for MAX.
3. Record the value of every internal node in the table.
4. State the optimal first move for MAX and justify it using the values of the root's children.

| Node | Minimax value |
|---|---:|
| A | |
| B | |
| C | |
| R | |

**Explanation:**  
_Write your reasoning here._


In [4]:
GAME_TREE = {
    'R': ['A', 'B', 'C'],
    'A': ['A1', 'A2'],
    'B': ['B1', 'B2'],
    'C': ['C1', 'C2'],
    'A1': ['A1a', 'A1b'],
    'A2': ['A2a', 'A2b'],
    'B1': ['B1a', 'B1b'],
    'B2': ['B2a', 'B2b'],
    'C1': ['C1a', 'C1b'],
    'C2': ['C2a', 'C2b'],
}

UTILITY = {
    'A1a': 3, 'A1b': 5,
    'A2a': 6, 'A2b': 9,
    'B1a': 1, 'B1b': 2,
    'B2a': 0, 'B2b': -1,
    'C1a': 7, 'C1b': 4,
    'C2a': 5, 'C2b': 6,
}
# GAME_TREE = {
#     'R': ['B', 'C', 'D'],
#     'B': ['B1', 'B2', 'B3'],
#     'C': ['C1', 'C2', 'C3'],
#     'D': ['D1', 'D2', 'D3'],
# }

# UTILITY = {
#     'B1': 3, 'B2': 12, 'B3': 8,
#     'C1': 2, 'C2': 4, 'C3': 6,
#     'D1': 14, 'D2': 5, 'D3': 2,
# }

def minimax(node, maximizing_player=True):
    """Return the minimax value of node.

    Terminal utilities are stored in UTILITY. For non-terminal nodes,
    recursively alternate MAX and MIN according to maximizing_player.
    """

    # Terminal node
    if node in UTILITY:
        return UTILITY[node]

    # Get children
    children = GAME_TREE[node]

    # MAX player
    if maximizing_player:
        return max(
            minimax(child, False)
            for child in children
        )

    # MIN player
    else:
        return min(
            minimax(child, True)
            for child in children
        )


def minimax_decision(root='R'):
    """Return (best_child, value) for MAX at the root.

    If root actions tie, retain the first action in GAME_TREE[root].
    """

    best_child = None
    best_value = float('-inf')

    for child in GAME_TREE[root]:
        value = minimax(child, False)

        # Use > so that the first child is retained in case of tie
        if value > best_value:
            best_value = value
            best_child = child

    return best_child, best_value


try:
    move, value = minimax_decision('R')
    print('Best root move:', move)
    print('Minimax value :', value)
except NotImplementedError:
    print('Implement minimax() and minimax_decision() before running this test.')

Best root move: C
Minimax value : 6


## Part 2 — Alpha–Beta Pruning

Alpha–Beta pruning returns the same decision as Minimax while skipping branches that cannot affect the final decision.

- `alpha`: best value MAX can guarantee so far;
- `beta`: best value MIN can guarantee so far;
- prune when `alpha >= beta`.

For this part, preserve the child ordering in `GAME_TREE`. Count each terminal utility that is actually evaluated and count each cutoff event.

### Alpha–Beta search visual

```text
MAX node: keep the largest value seen so far  → alpha
MIN node: keep the smallest value seen so far → beta

                 MAX (R)
               /    |    \
            MIN(A) MIN(B) MIN(C)
              ...   ...   ...

Cut off the remaining children of a node when alpha >= beta.
The cutoff skips work; it does not change the minimax value.
```

The exact leaves visited depend on the fixed left-to-right child order in `GAME_TREE`.


### Task 2 — Implement and analyse Alpha–Beta pruning

1. Implement Alpha–Beta pruning.
2. List the terminal leaves that were evaluated and the branches that were pruned.
3. Compare the root decision with your Minimax result from Part 1.
4. Explain why pruning does not change the optimal decision.

| Measure | Minimax | Alpha–Beta |
|---|---:|---:|
| Optimal move | | |
| Root value | | |
| Terminal leaves evaluated | | |
| Branches pruned | 0 | |

**Explanation:**  
_Write your reasoning here._


**Before coding:** use the rules and visual above to predict at least one legal move and the expected terminal condition.

In [5]:
import math


def alpha_beta(node, alpha=-math.inf, beta=math.inf,
               maximizing_player=True, stats=None):
    if stats is None:
        stats = {'leaves': 0, 'prunes': 0}

    if node in UTILITY:
        stats['leaves'] += 1
        return UTILITY[node], stats

    children = GAME_TREE[node]

    if maximizing_player:
        value = -math.inf
        for child in children:
            child_value, stats = alpha_beta(
                child, alpha, beta, maximizing_player=False, stats=stats
            )
            value = max(value, child_value)
            alpha = max(alpha, value)
            if alpha >= beta:
                stats['prunes'] += 1
                break
        return value, stats

    value = math.inf
    for child in children:
        child_value, stats = alpha_beta(
            child, alpha, beta, maximizing_player=True, stats=stats
        )
        value = min(value, child_value)
        beta = min(beta, value)
        if alpha >= beta:
            stats['prunes'] += 1
            break
    return value, stats



def alpha_beta_decision(root='R'):
    """Return (best_child, value, stats) for MAX at the root."""

    stats = {'leaves': 0, 'prunes': 0}

    best_child = None
    best_value = -math.inf
    alpha = -math.inf
    beta = math.inf

    for child in GAME_TREE[root]:
        child_value, stats = alpha_beta(
            child,
            alpha,
            beta,
            maximizing_player=False,
            stats=stats
        )

        # Use > so the first child is retained in case of a tie
        if child_value > best_value:
            best_value = child_value
            best_child = child

        alpha = max(alpha, best_value)

    return best_child, best_value, stats


try:
    move, value, stats = alpha_beta_decision('R')
    print('Best move      :', move)
    print('Value          :', value)
    print('Leaves examined:', stats['leaves'])
    print('Prunes         :', stats['prunes'])
except NotImplementedError:
    print('Implement alpha_beta() and alpha_beta_decision() before running this test.')


Best move      : C
Value          : 6
Leaves examined: 9
Prunes         : 2


## Part 3 — Standard Nim

In **standard Nim**, the game state is a tuple of pile sizes. On each turn, a player chooses exactly one non-empty pile and removes any positive number of objects from that pile. The player who removes the last object wins.

For this task, MAX moves first from the state `(3, 2)`. Implement the legal-move generator and a solver that determines whether a state is winning or losing for the player whose turn it is.

Use the following convention:

- `True`: the current player has a winning strategy;
- `False`: the current player loses under optimal play.

Sort states in a consistent way so that output is reproducible.

### Standard Nim — pile visual and rules

```text
Starting position:  (3, 2)
Pile 1:  ● ● ●       Pile 2:  ● ●

A legal move changes ONE pile only, removing at least one object:
(3, 2) → (1, 2)   is legal (remove two from pile 1)
(3, 2) → (3, 0)   is legal (remove two from pile 2; zero pile is omitted → (3,))
(3, 2) → (2, 1)   is NOT legal in one move (both piles changed)
```

**End rule:** the player who takes the last object wins. A state with no objects is therefore losing for the player whose turn it is.
#  Nim Game: Rules and Visual Game Tree

## 1. Rules of the Game

Nim is a two-player strategy game played with piles of objects.

* Players take turns removing objects from a pile.
* On each turn, a player must remove at least one object.
* A player may remove at most two objects in this example.
* A player who removes the last object wins.
* Both players play optimally.

**Objective:** Determine the winning strategy using adversarial search.

## 2. Example: Nim with One Pile

Suppose the initial pile contains two objects, and each player can remove one or two objects per turn.

**Initial state:** `[2]`

The available moves are:

* Remove one object: `[2] → [1]`
* Remove two objects: `[2] → [0]`

## 3. Visual Game Tree

```text
                  MAX
                 [2]
                /   \
         Remove 1   Remove 2
              /       \
            MIN       MIN
            [1]       [0]
             |          |
          Remove 1   Game Over
             |       MAX wins
            [0]
             |
          Game Over
          MIN wins
```

### 4. Understanding the Tree

* **MAX:** The player trying to maximize the outcome.
* **MIN:** The opponent trying to minimize MAX's outcome.
* **Terminal state:** A state in which no objects remain and the game ends.
* **Optimal play:** Each player selects the move that gives them the best possible outcome.

### 5. Think Before Coding

1. What moves are available when the pile contains three objects?
2. Which player wins when the initial pile contains two objects?
3. How can you represent a game state in Python?
4. How can minimax determine the best move?
5. What is the base case for the recursive function?

**Task:** Implement the required Nim game logic in the coding cell below. Follow the function signature and requirements provided in the assignment.



### Task 3 — Implement and analyse Standard Nim

1. Generate all legal moves from `(3, 2)`.
2. Determine which resulting states are winning or losing for the next player.
3. Identify every winning first move for MAX.
4. Explain why removing one object from the pile of size 3 is, or is not, optimal.
5. Test your implementation on `(1,)`, `(1, 1)`, `(2, 2)`, and `(1, 2)`.

| Starting state | Winning or losing for player to move? | Winning move(s), if any |
|---|---|---|
| `(3, 2)` | | |
| `(1,)` | | |
| `(1, 1)` | | |
| `(2, 2)` | | |
| `(1, 2)` | | |

**Explanation of MAX's strategy from `(3, 2)`:**  
_Write your reasoning here._


**Before coding:** use the rules and visual above to predict at least one legal move and the expected terminal condition.

In [3]:
from functools import lru_cache


def nim_successors(state):
    """Return all distinct legal next states for standard Nim.

    A move removes a positive number of objects from exactly one pile.
    Represent each state as a tuple sorted in non-increasing order;
    omit zero-sized piles.
    """
    successors = set()

    for i, pile in enumerate(state):
        # Remove 1, 2, ..., pile objects
        for remaining in range(pile):
            new_state = list(state)

            if remaining == 0:
                # This pile becomes empty
                new_state.pop(i)
            else:
                new_state[i] = remaining

            # Sort non-increasingly and remove zeros
            new_state = tuple(
                sorted(
                    (x for x in new_state if x > 0),
                    reverse=True
                )
            )

            successors.add(new_state)

    return sorted(successors, reverse=True)


@lru_cache(maxsize=None)
def nim_is_winning(state):
    """Return True if the current player has a winning strategy.

    A state with no legal moves is losing.
    """

    successors = nim_successors(state)

    # No legal moves -> losing position
    if not successors:
        return False

    # Winning if there exists a move to a losing state
    return any(not nim_is_winning(next_state)
               for next_state in successors)


def nim_best_moves(state):
    """Return all legal moves that leave a losing state for the opponent."""

    return [
        next_state
        for next_state in nim_successors(state)
        if not nim_is_winning(next_state)
    ]


initial_nim_state = (3, 2)

try:
    print('Initial state:', initial_nim_state)
    print('Winning for player to move:', nim_is_winning(initial_nim_state))
    print('Winning moves:', nim_best_moves(initial_nim_state))
except NotImplementedError:
    print('Implement the Nim functions before running this test.')

Initial state: (3, 2)
Winning for player to move: True
Winning moves: [(2, 2)]


## Part 4 — Grundy's Game

Grundy's Game is different from standard Nim. A move selects one pile and splits it into **two non-empty piles of unequal sizes**. A player who cannot make a legal move loses.

Start from the single pile `(5,)`. Represent states as tuples sorted in non-increasing order. Piles of equal size are not a legal result of a split, and zero-sized piles are not included.

### Grundy's Game — pile-splitting visual and rules
# Part 4 — Grundy's Game

Grundy's Game is different from standard Nim. A move selects one pile and splits it into **two non-empty piles of unequal sizes**. A player who cannot make a legal move loses.

Start from the single pile `(5,)`. Represent states as tuples sorted in non-increasing order. Piles of equal size are not a legal result of a split, and zero-sized piles are not included.

## 1. Rules of the Game

* Players take turns selecting one existing pile.
* The selected pile must be split into two non-empty piles of unequal sizes.
* A split such as `5 → 1 + 4` is legal.
* A split such as `4 → 2 + 2` is illegal because the resulting piles have equal sizes.
* A split such as `5 → 0 + 5` is illegal because both resulting piles must be non-empty.
* After every move, represent the state as a tuple sorted in non-increasing order.
* Piles of size `1` cannot be split.
* A player who cannot make a legal move loses.
* Both players play optimally.

**Objective:** Use adversarial search to determine winning and losing states and identify the optimal strategy.

## 2. Pile-Splitting Visual
```text
One pile of 5 objects:
● ● ● ● ●

Legal unequal split examples:
5 → 1 + 4       gives state (4, 1)
5 → 2 + 3       gives state (3, 2)

Not allowed:
5 → 0 + 5       (a pile must split into two non-empty piles)
4 → 2 + 2       (the two new piles must have unequal sizes)
```

Unlike standard Nim, you do **not** remove objects from a pile: you replace one pile by two unequal, non-empty piles. Piles are sorted in non-increasing order in the tuple representation, so `(4, 1)` is the same state as `(1, 4)` and is represented as `(4, 1)`.
Consider a pile containing five objects.

**Initial state:** `(5,)`

```text
                ONE PILE
              ● ● ● ● ●
                   |
             Split into two
             unequal piles
              /         \
             /           \
        1 + 4             2 + 3
          |                 |
        (4, 1)            (3, 2)
```

The two legal moves are:

* `5 → 1 + 4`, producing `(4, 1)`.
* `5 → 2 + 3`, producing `(3, 2)`.

**Important:** The state `(4, 1)` represents the same arrangement as `(1, 4)`. We always store the larger pile first.

## 3. Visual Game Tree

Starting from `(5,)`, the complete first level of the game tree is:

```text
                         MAX
                         (5,)
                       /       \
                  Split         Split
                  1 + 4         2 + 3
                    /             \
                  MIN             MIN
                (4, 1)           (3, 2)
                /     \           /   \
             Split   Split      Split  Split
               ...     ...       ...    ...
```

Each subsequent move selects one pile and replaces it with two unequal, non-empty piles. All other piles remain unchanged.

For example, from `(4, 1)`:

```text
                    (4, 1)
                   /       \
                Split 4   Split 1
                 /   \        |
              1 + 3  4 → ?   No move
                |
             (3, 1, 1)
```

The legal split of the pile of size `4` is `1 + 3`. The split `2 + 2` is illegal. The pile of size `1` cannot be split.

The resulting state is `(3, 1, 1)`.

From `(3, 2)`, the legal splits include:

* `3 → 1 + 2`, producing `(2, 2, 1)`.
* `2` cannot be split into two unequal, non-empty piles.

The split `3 → 1 + 2` is legal, but the resulting state `(2, 2, 1)` contains two piles of size `2`. These equal-sized piles are allowed to exist in a state; the rule prohibits splitting a pile into two equal-sized piles, not having equal-sized piles already present.

## 4. Understanding Winning and Losing States

* **MAX:** The player whose turn it is at the root of the search tree.
* **MIN:** The opponent, who also chooses moves optimally.
* **Game state:** A tuple such as `(5,)`, `(4, 1)`, or `(3, 2)`.
* **Legal move:** Replacing one pile with two unequal, non-empty piles.
* **Terminal state:** A state in which no pile can be split legally.
* **Winning state:** A state from which the current player can force a win against an optimal opponent.
* **Losing state:** A state from which every legal move allows the opponent to force a win.
* **Optimal play:** Each player selects a move that gives them the best possible outcome.

A terminal state is a **losing state for the player whose turn it is**, because that player has no legal move.

## 5. Representing States in Python

Examples of valid states:

```python
(5,)          # One pile of size 5
(4, 1)        # Two piles: 4 and 1
(3, 2)        # Two piles: 3 and 2
(2, 2, 1)     # Three piles: 2, 2, and 1
(1, 1, 1, 1)  # Four piles of size 1
```

The following representations are not valid under the assignment's conventions:

```python
(0, 5)        # Zero-sized piles are not included
(1, 4)        # Not in non-increasing order
```

Remember that a state may contain equal-sized piles, but an individual move must never split a pile into two equal-sized piles.

## 6. Think Before Coding

1. What are all the legal moves from the initial state `(5,)`?
2. Why is splitting `5` into `2 + 3` legal, while splitting `4` into `2 + 2` is illegal?
3. What states can result from splitting the pile of size `4` in `(4, 1)`?
4. Why can the pile of size `1` never be selected for a move?
5. Why must states be stored in non-increasing order?
6. What is the base case for a recursive winning-state function?
7. How can memoization avoid evaluating the same state repeatedly?

**Task:** Implement the required Grundy's Game logic in the coding cell below. Follow the existing function signatures, state representation, and requirements provided in the assignment.




### Task 4 — Implement and analyse Grundy’s Game

1. List every legal first move from `(5,)`.
2. Evaluate each resulting state as winning or losing for the player to move.
3. Identify MAX's winning first move, if one exists.
4. Draw or describe one complete line of play that demonstrates the winning strategy.
5. Explain why splitting a pile of size 2 into `(1, 1)` is illegal in this game.
6. Compare the move rule in Grundy's Game with standard Nim.

| State | Legal successor states | Winning or losing for player to move? |
|---|---|---|
| `(5,)` | | |
| `(1, 4)` | | |
| `(2, 3)` | | |
| `(1, 1, 3)` | | |
| `(1, 2, 2)` | | |

**Explanation of MAX's strategy:**  
_Write your reasoning here._


**Before coding:** use the rules and visual above to predict at least one legal move and the expected terminal condition.

In [4]:
from functools import lru_cache


@lru_cache(maxsize=None)
def grundy_successors(state):
    """Return all distinct legal states after one Grundy's Game move.

    Select one pile n and replace it by a+b where:
      - a + b = n
      - a >= 1 and b >= 1
      - a != b

    Return each resulting state as a tuple sorted in non-increasing order.
    """
    successors = set()

    for i, n in enumerate(state):
        # Split n into a + b
        for a in range(1, n):
            b = n - a

            # Equal split is not allowed
            if a == b:
                continue

            new_state = list(state)

            # Remove the original pile
            new_state.pop(i)

            # Add the two new piles
            new_state.extend([a, b])

            # Sort in non-increasing order
            new_state = tuple(sorted(new_state, reverse=True))

            successors.add(new_state)

    return tuple(sorted(successors, reverse=True))


@lru_cache(maxsize=None)
def grundy_is_winning(state):
    """Return True if the current player has a winning strategy.

    A state with no legal moves is losing.
    """

    successors = grundy_successors(state)

    # No legal moves -> losing state
    if not successors:
        return False

    # Winning if there is at least one move to a losing state
    return any(
        not grundy_is_winning(next_state)
        for next_state in successors
    )


def grundy_best_moves(state):
    """Return all legal next states that leave a losing state for the opponent."""

    return tuple(
        next_state
        for next_state in grundy_successors(state)
        if not grundy_is_winning(next_state)
    )


initial_grundy_state = (5,)

try:
    print('Initial state:', initial_grundy_state)
    print('Legal first moves:', grundy_successors(initial_grundy_state))
    print('Winning for player to move:', grundy_is_winning(initial_grundy_state))
    print('Winning moves:', grundy_best_moves(initial_grundy_state))
except NotImplementedError:
    print('Implement the Grundy’s Game functions before running this test.')

Initial state: (5,)
Legal first moves: ((4, 1), (3, 2))
Winning for player to move: True
Winning moves: ((4, 1),)


## Quick Python guide — `@lru_cache` and function arguments

The Nim and Grundy's Game solvers use recursion. The same state can be reached by different move sequences, so a solver may otherwise recompute the answer many times. `functools.lru_cache` remembers results for previously seen inputs.

### 1. What does `@lru_cache(maxsize=None)` mean?

```python
from functools import lru_cache

@lru_cache(maxsize=None)
def is_winning(state):
    if not successors(state):
        return False
    return any(not is_winning(next_state) for next_state in successors(state))
```

- `@lru_cache(...)` is a **decorator**: it wraps the function with a cache while keeping the same way of calling it.
- `maxsize=None` means keep all cached results (unbounded cache). `maxsize=128`, for example, keeps up to 128 recent results and may evict older ones.
- On the first call such as `is_winning((3, 2))`, Python computes the result and stores it. A later call with the same argument returns the stored result instead of running the function body again.
- Arguments used as cache keys must be **hashable**. Tuples of integers such as `(3, 2)` are hashable; a list such as `[3, 2]` is not. That is why this lab represents states as tuples.
- Cache keys include the function's arguments. If a cached function accepts several arguments, each argument combination can have a separate entry.

### 2. What can be passed to `lru_cache`?

`@lru_cache(maxsize=128, typed=False)` accepts these options:

- `maxsize`: maximum number of stored calls; use `None` for no limit, or a non-negative integer such as `128`.
- `typed`: if `True`, arguments of different types are cached separately in cases where they compare equal (for example, `1` and `1.0`). The default is `False`.

Use it on functions whose output depends on their arguments and that can safely reuse old results. Avoid it for functions that depend on changing external state or that must perform side effects every time.

### 3. Inspecting and clearing the cache

```python
is_winning((3, 2))
print(is_winning.cache_info())  # hits, misses, maxsize, current size
is_winning.cache_clear()       # remove all stored results
```

`cache_info()` is useful for seeing whether repeated states are being reused. `cache_clear()` is useful when you want to rerun an experiment from a clean cache.

### 4. Understanding the special-looking function syntax in this lab

- `def nim_is_winning(state):` defines a function with one parameter, `state`.
- `nim_is_winning((3, 2))` passes a tuple as the argument. Parentheses around the tuple matter: `(3, 2)` is a tuple; `(3)` is just the integer `3`; `(3,)` is a one-item tuple.
- `def alpha_beta(node, alpha=-math.inf, beta=math.inf, maximizing_player=True, stats=None):` has required parameter `node` and optional parameters with defaults. You can call `alpha_beta('R')`, or override values by keyword, e.g. `alpha_beta('R', alpha=-math.inf, beta=math.inf, maximizing_player=True, stats={...})`.
- `*args` (if you encounter it) collects extra positional arguments into a tuple; `**kwargs` collects extra keyword arguments into a dictionary. They are not needed for the required functions in this lab.
- `None` is commonly used to mean “no value supplied yet”; the Alpha–Beta starter uses `stats=None` and creates a fresh dictionary inside the function.

**Important:** `lru_cache` stores results based on inputs, so don't mutate a cached return value. In this lab the cached functions return booleans or tuples, which are safe choices.
